# Assignment 6: Extracting modes of climate variability with PCA/EOF

Climate fields are high-dimensional and strongly correlated in space. A global sea surface
temperature grid has thousands of points, but the temperature at any one of them tells you a
great deal about its neighbours, so the field has far fewer independent degrees of freedom
than it has grid cells. This is exactly the situation principal component analysis was built
for.

In climate science, the principal components of a spatial field are called **empirical
orthogonal functions** (EOFs), and the leading ones frequently correspond to named, physically
meaningful modes of variability. In this assignment you will recover one of the most important
of them (**El Nino-Southern Oscillation**) from raw sea surface temperature data, without
ever telling the algorithm what to look for.

The data is NOAA's Extended Reconstructed SST version 5, monthly means from 1854 to present,
streamed over OPeNDAP so you never download the whole file.

Answer each numbered question in the empty cell below it.

In [ ]:
!pip install xarray netcdf4

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

## Load the sea surface temperature field

This is the same dataset used in the PCA lecture notes.

In [ ]:
sst = xr.open_dataset("http://www.esrl.noaa.gov/psd/thredds/dodsC/Datasets/noaa.ersst.v5/sst.mnmean.nc")
sst

We restrict to the **tropical Pacific** (120E-280E, 20S-20N) and to 1950 onwards, where
observational coverage is good. ENSO is a tropical Pacific phenomenon, so this is where its
signature is clearest.

In [ ]:
tropical = sst.sst.sel(lat=slice(20, -20), lon=slice(120, 280),
                       time=slice("1950-01-01", "2020-12-31"))
print(tropical.shape, "  (time, lat, lon)")
tropical.isel(time=0).plot(figsize=(10, 3.2), cmap="RdYlBu_r")
plt.title("SST, January 1950")
plt.show()

## Part 1: Anomalies

PCA finds directions of maximum variance. Applied to raw SST, the leading mode would simply
be the seasonal cycle, which is large, obvious, and not what we are looking for. We remove it
first.

1) Compute monthly SST **anomalies**: subtract the long-term mean for each calendar month
from each observation. `groupby("time.month")` will do this in one line.

2) Plot the anomaly field for January 1998 and for January 1989. These are a strong El Nino and a strong La Nina respectively. Describe how they differ.

3) Explain why we remove the seasonal cycle before running PCA, and what the leading mode would have been if we had not.

*Write your answer here.*

## Part 2: Prepare the data matrix

PCA expects a 2D matrix of samples by features. Here each **time step** is a sample and each
**grid cell** is a feature, so we reshape from (time, lat, lon) to (time, lat x lon).

Ocean grids contain land, which appears as `NaN`. Those columns must be removed before PCA
and put back afterwards.

4) Reshape the anomaly field into a 2D array of shape (n_times, n_gridcells). Identify and
drop the columns that are entirely `NaN`. Report the shape before and after.

5) Grid cells near the poles represent smaller areas than cells at the equator, so an unweighted PCA over-weights high latitudes. Multiply each column by the square root of the cosine of its latitude to correct for this. Explain why the *square root* is the right weight when PCA works on squared deviations.

## Part 3: Compute the EOFs

6) Fit a `PCA` retaining the first 10 components. Report the fraction of variance explained
by each.

7) Plot the explained variance spectrum. How many modes stand clearly above the rest, and how much of the total variance does the leading mode alone account for?

## Part 4: Interpret the leading mode

8) Reshape the first EOF (the first row of `pca.components_`) back onto the lat-lon grid,
remembering to reinsert the land cells as `NaN`. Plot it as a map.

9) Describe the spatial pattern. Where are the largest loadings, and what is the sign structure across the basin? Compare it to what you saw in the 1998 anomaly map from question 2.

*Write your answer here.*

10) Plot the first **principal component** (the time series `pca.transform(X)[:, 0]`) against
time. This is how strongly the leading spatial pattern is expressed in each month.

## Part 5: Validate against an independent index

The claim "EOF 1 is ENSO" needs testing against something the PCA never saw.

The **Nino 3.4 index** is the standard ENSO measure: the mean SST anomaly over
170W-120W, 5S-5N. Compute it directly from the same dataset.

11) Compute the Nino 3.4 index from your anomaly field and plot it alongside PC1 on the same
axes. You may need to normalize both to compare them, and PC1's sign is arbitrary. If it is
inverted, flip it and say so.

12) Compute the correlation between PC1 and the Nino 3.4 index. How strong is it?

13) Identify the five largest El Nino events in your PC1 time series. Do they match the historically recognized events (1982-83, 1997-98, 2015-16 are the big three)?

## Part 6: Reconstruction and limitations

14) Reconstruct the anomaly field using only the first mode, then using the first five. Plot
the reconstruction of January 1998 in both cases alongside the original. What is captured,
and what is lost?

15) Compute the RMSE of the reconstruction against the original field as a function of the number of modes retained. Plot it.

16) PCA imposes three constraints that have no physical justification: the modes must be
**orthogonal**, the relationship must be **linear**, and **variance is treated as
importance**. For each, give one sentence on why a real climate mode might violate it.

*Write your answer here.*

17) EOF 2 of tropical Pacific SST is often interpreted as ENSO's decay phase rather than an independent mode. Given the orthogonality constraint, explain why you should be cautious about assigning physical meaning to the second and higher EOFs.

*Write your answer here.*